In [ ]:
import cv2
import numpy as np 
import pandas as pd
from skimage.feature import graycomatrix, graycoprops
import os
from skimage.measure import shannon_entropy
from skimage.feature import local_binary_pattern
from scipy.stats import skew

#local binary pattern of image
def loc_bin_mean(img):
    radius = 1
    n_points = 8*radius

    lbp = local_binary_pattern(img , n_points,radius,method ='uniform')
    return np.mean(lbp)

#fft energy of feature 
def compute_fft(image):
    f = np.fft.fft2(image)             
    fshift = np.fft.fftshift(f)        
    magnitude = np.log(1 + np.abs(fshift))  
    energy = np.sum(magnitude ** 2)
    return energy

#high freq ratio
def high_frequency_ratio(img):
    f = np.fft.fft2(img)
    fshift = np.fft.fftshift(f)
    magnitude = np.abs(fshift)

    center = magnitude[
        magnitude.shape[0]//4:3*magnitude.shape[0]//4,
        magnitude.shape[1]//4:3*magnitude.shape[1]//4
    ]

    high_freq = magnitude.sum() - center.sum()
    ratio = high_freq / magnitude.sum()

    return ratio

#std dev of noise in image
def get_noise(img):
    blur = cv2.GaussianBlur(img, (5,5), 0)
    noise = img - blur
    std = np.std(noise)
    return std
#entropy of image 
def get_entropy(img):
    return shannon_entropy(img)

#edge density and strength
def edge_analysis(img):
    edges = cv2.Canny(img, 150, 250)
    edge_density = np.sum(edges > 0) / edges.size
    sobelx = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=3)
    
    magnitude = np.sqrt(sobelx**2 + sobely**2)
    return edge_density,np.mean(magnitude)

#eggb variance of image
def get_rgb_variance(img):
    r,g,b = cv2.split(img)

    var = np.var(r) + np.var(g) + np.var(b)
    return var

def glcm_features(img):
    # Convert to 8-bit (important)
    img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype('uint8')
    
    # Compute GLCM
    glcm = graycomatrix(
        img,
        distances=[1],
        angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
        levels=256,
        symmetric=True,
        normed=True
    )
    
    # Extract features
    contrast = graycoprops(glcm, 'contrast').mean()
    homogeneity = graycoprops(glcm, 'homogeneity').mean()
    correlation = graycoprops(glcm, 'correlation').mean()
    energy = graycoprops(glcm, 'energy').mean()
    dissimilarity = graycoprops(glcm, 'dissimilarity').mean()
    
    return contrast, homogeneity,correlation,energy,dissimilarity

def laplacian_analysis(img):
    # Apply Laplacian
    lap = cv2.Laplacian(img, cv2.CV_64F)
    
    # Compute variance (key feature)
    lap_var = np.var(lap)
    
    return lap_var
#sharpness
def sharpness(img):
    return np.mean(cv2.Laplacian(img, cv2.CV_64F) ** 2)
#color features
def color_features(img):
    r, g, b = cv2.split(img)

    mean_r = np.mean(r)
    mean_g = np.mean(g)
    mean_b = np.mean(b)

    std_r = np.std(r)
    std_g = np.std(g)
    std_b = np.std(b)

    return mean_r, mean_g, mean_b, std_r, std_g, std_b
#color correlation
def color_correlation(img):
    r, g, b = cv2.split(img)

    rg = np.corrcoef(r.flatten(), g.flatten())[0, 1]
    rb = np.corrcoef(r.flatten(), b.flatten())[0, 1]
    gb = np.corrcoef(g.flatten(), b.flatten())[0, 1]

    return rg, rb, gb

def histogram_skew(img):
    return skew(img.flatten())
#extracct the impo features from the images
def features_extraction(image):
    img = cv2.imread(image)
    if img is None:
        return None
    color_img = img.copy()
    img = cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)

    #local binary pattern
    lbp = loc_bin_mean(img)

    #fft energy
    fft_energy = compute_fft(img)

    #hf ration
    hf_ratio = high_frequency_ratio(img)

    #entropy of image
    entropy = get_entropy(img)

    #rgb variance of image
    rgb_variance = get_rgb_variance(color_img)

    #noise of the image
    noise_std = get_noise(img)

    #edge density and strength of the image
    edge_density,edge_strength = edge_analysis(img)

    #contrast and homogeneity of the image
    contrast, homogeneity ,correlation , energy,dissimilarity = glcm_features(img)

    #laplacian variance of the image
    laplacian_variance = laplacian_analysis(img)

    #sharpness
    sharp = sharpness(img)

    #histogram skew
    skewness = histogram_skew(img)

    #mean and std dev of rgb
    mean_r, mean_g, mean_b, std_r, std_g, std_b = color_features(color_img)

    #color correlation 
    color_corr_rg, color_corr_rb, color_corr_gb = color_correlation(color_img)


    return [
            lbp,
            fft_energy,
            hf_ratio,
            entropy,
            noise_std,
            edge_density,
            edge_strength,
            contrast,
            homogeneity,
            correlation,
            energy,
            dissimilarity,
            laplacian_variance,
            sharp,
            skewness,
            mean_r, mean_g, mean_b,
            std_r, std_g, std_b,
            color_corr_rg,
            color_corr_rb,
            color_corr_gb
        ]


image_dataset = []
real_image_folder = 'RealArt'
ai_image_folder='AiArtData'


for file in os.listdir(real_image_folder):
    path = os.path.join(real_image_folder,file)

    features = features_extraction(path)

    if features is not None:
        features.append("Real")
        image_dataset.append(features)


for file in os.listdir(ai_image_folder):
    path = os.path.join(ai_image_folder,file)

    features = features_extraction(path)

    if features is not None:
        features.append("AI")
        image_dataset.append(features)

columns = [
    'lbp',
    'fft_energy',
    'high_freq_ratio',
    'entropy',
    'noise_std',
    'edge_density',
    'edge_strength',
    'contrast',
    'homogeneity',
    'correlation',
    'energy',
    'dissimilarity',
    'laplacian_variance',
    'sharpness',
    'histogram_skew',
    'mean_r', 'mean_g', 'mean_b',
    'std_r', 'std_g', 'std_b',
    'color_corr_rg',
    'color_corr_rb',
    'color_corr_gb',
    'label'
]

df = pd.DataFrame(image_dataset,columns = columns)

df = df.sample(frac= 1,random_state=42).reset_index(drop=True)

df.to_csv("Image_features_Dataset.csv",index = False)

print("Dataset is created!!!!")
   